In [1]:
import pyspark.sql, pyspark.sql.functions
from pyspark.sql import SparkSession

# Стартиране на SparkSession 
spark = (
    SparkSession.builder
    .appName("Test")
    .master("local[*]")
    .getOrCreate()
)

print("SparkSession стартиран успешно")
print("Spark версия:", spark.version)

SparkSession стартиран успешно
Spark версия: 4.0.1


In [3]:
from pyspark.sql.functions import col

# Примерен DataFrame с 1 млн. реда
data = [(i, i*2) for i in range(1_000_000)]
df = spark.createDataFrame(data, ["id", "value"])

# Разпределяне на DataFrame върху 4 партиции
df = df.repartition(4)

# Проверка на броя партиции
print("Брой партиции:", df.rdd.getNumPartitions())

# Примерна трансформация: филтриране
df_filtered = df.filter(col("value") > 500_000)

# Преглед на резултат (няколко реда)
df_filtered.show(5)


Брой партиции: 4
+------+------+
|    id| value|
+------+------+
|373042|746084|
|302142|604284|
|291373|582746|
|274720|549440|
|334869|669738|
+------+------+
only showing top 5 rows


In [4]:
# Път към CSV файла
file_path = "customers-data.csv"

# Зареждане на CSV в DataFrame
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Показване на записите
df.show()

+-----------+---------------+----+---------------+----------------+-------------+
|customer_id|           name| age|purchase_amount|product_category|purchase_date|
+-----------+---------------+----+---------------+----------------+-------------+
|          1|    Ivan Petrov|  25|            100|     Electronics|   2025-01-05|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|
|          3|Georgi Georgiev|  40|            150|           Books|   2025-01-07|
|          4|    Anna Koleva|  -5|             50|     Electronics|   2025-01-08|
|          5|           NULL|  28|           NULL|        Clothing|   2025-01-09|
|          6| Petar Dimitrov|  35|            300|           Books|   2025-01-10|
|          7|    Ivan Petrov|NULL|            120|     Electronics|   2025-01-11|
|          8|  Maria Ivanova|  32|            250|        Clothing|         NULL|
|          9|Geo

In [15]:
from pyspark.sql.functions import sum as _sum, avg

# Задаваме партициониране (например 2 партиции)
df = df.repartition(2)

# Обобщаване - сума и средна на purchase_amount
df_summary = df.agg(
    _sum("purchase_amount").alias("total_purchase"),
    avg("purchase_amount").alias("avg_purchase")
)

# Показваме резултата
df_summary.show()

# Проверка на броя партиции
print("Number of partitions:", df.rdd.getNumPartitions())

+--------------+------------------+
|total_purchase|      avg_purchase|
+--------------+------------------+
|          1550|172.22222222222223|
+--------------+------------------+

Number of partitions: 2
